In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from scipy import stats
from scipy.stats import qmc   # for Latin Hypercube
import warnings
warnings.filterwarnings('ignore')
import os


In [2]:

path='/Users/mgabr001/Desktop/CarbonCopies/NetmorphParOptim/'

os.chdir(path)

# Parameters under investigation (Min-Max, # of bins): TOTAL par space = 6 * 15 * 15 * 6 * 11 * 4 * 10 = 3,564,000
# days                20 - 25    # 6
# pyramidal           16 - 128   # 15
# interneuron         16 - 128   # 15
# minneuronseparation 10 - 15    # 6
# shape.radius        100 - 200  # 11
# shape.thickness     20 - 50    # 4
# dm.weight           0.1 - 1.0  # 10

l_bounds = np.array([20, 16, 16, 10, 100, 20, 0.1 ])
# Upper bounds are selected 1 increment higher than the required range, since the lower bound of the bin (lb, lb+inc) is being assigned 
# (for example, all values between (25, 26) are assigned 25 which is the required max value for days)   
u_bounds = np.array([26, 136, 136, 16, 210, 60, 1.1]) 
increments = np.array([1, 8, 8, 1, 10, 10, 0.1 ])

sampler = qmc.LatinHypercube(d=7, rng = 2727)   # fix random number generator seed for reproducability 
sample = sampler.random(n=700) 

scaled_smp = qmc.scale(sample, l_bounds, u_bounds)

# qmc.discrepancy(sample)
print(scaled_smp[:5,:])

for j in range(scaled_smp.shape[1]):
    lb = l_bounds[j]
    ub = l_bounds[j]+increments[j]
    inc = increments[j]

    bins = (u_bounds - l_bounds) / increments
    
    for k in range( int(bins[j]) ):
        for i in range(scaled_smp.shape[0]):
            if scaled_smp[i,j] > lb and scaled_smp[i,j] < ub:
                scaled_smp[i,j] = lb
        lb = ub
        ub = lb + inc

print(scaled_smp[:5,:])



[[ 24.11704995  27.45933059  55.30793563  14.75363705 125.51850055
   32.39209019   0.32786115]
 [ 20.61279089 103.81649398 129.31143981  15.8372549  173.437659
   30.02483383   0.73167551]
 [ 24.55254718 120.50360503  31.15866401  15.97847498 162.26741276
   26.74851858   0.74414687]
 [ 20.86455435 106.7356689   52.24297953  10.43099349 135.15429347
   52.91904492   0.58529761]
 [ 25.73848672 103.73506527  30.74086448  15.20318347 114.45548086
   42.93961353   0.94673813]]
[[ 24.   24.   48.   14.  120.   30.    0.3]
 [ 20.   96.  128.   15.  170.   30.    0.7]
 [ 24.  120.   24.   15.  160.   20.    0.7]
 [ 20.  104.   48.   10.  130.   50.    0.5]
 [ 25.   96.   24.   15.  110.   40.    0.9]]


In [3]:
scaled_smp.shape

(700, 7)

In [4]:
# Create Dataframe based from scaled_smp data.

cols = ['days','pyramidal','interneuron','minneuronseparation','shape.radius','shape.thickness','dm.weight ']
df = pd.DataFrame(data = scaled_smp, columns = cols)

df = df.astype({'days':'int'})
df = df.astype({'pyramidal':'int'})
df = df.astype({'interneuron':'int'})
df = df.astype({'minneuronseparation':'int'})
df = df.astype({'shape.radius':'int'})
df = df.astype({'shape.thickness':'int'})
df = df.astype({'dm.weight ':'float'})

# Drop Index inplace
df.reset_index(drop=True, inplace=True)
# write 700 samples into the excel file.
with pd.ExcelWriter("ParameterSpace_set1_700_samples.xlsx") as writer:
    df.iloc[:700].to_excel(writer, index=False)  
   
print(df.shape)
df.head(20)

(700, 7)


,days,pyramidal,interneuron,minneuronseparation,shape.radius,shape.thickness,dm.weight
0,24,24,48,14,120,30,0.3
1,20,96,128,15,170,30,0.7
2,24,120,24,15,160,20,0.7
3,20,104,48,10,130,50,0.5
4,25,96,24,15,110,40,0.9
5,23,56,104,14,120,40,0.9
6,23,16,112,14,200,20,0.6
7,22,64,64,11,160,40,0.2
8,20,64,24,14,170,50,0.5
9,23,32,32,12,180,40,0.8
